In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('/home/domans/dianne-codebase/')
import viewer
import dianne
import pandas as pd
import numpy as np
import matplotlib.colors as mcolors
dianne.setNotebookWidth(80)
from tqdm import tqdm
import pickle
import os

def createH2(slide, mpath=None):
    df_temp = pd.read_csv(f'{mpath}/{slide}.matrix-H.csv', header=None)
    df_temp.iloc[:2,:2] *= 0.25 / 0.2208187960959237
    sname = f'{mpath}/{slide}.matrix-H-2.csv'
    if not os.path.isfile(sname):
        df_temp.to_csv(sname, index=False, header=False)
    return

idm = './identity-matrix.csv'
pd.DataFrame([[1,0,0], [0,1,0], [0,0,1]]).to_csv(idm, index=False, header=False)

celltype_color_dict_v1 = {
    'Malignant OPC':        'green', 
    'Malignant NPC':        'lime',
    'Malignant AC':         'red',
    'Malignant Mes.':       'dodgerblue',
    'Malignant Hyp.':       'cyan',
    'Astrocyte':           'salmon',
    'Astrocyte Rea.':      'sandybrown',
    'Neuron':              'orange',
    'Oligodendrocyte':     'peachpuff',
    'Microglia':          'black',
    'Macrophage':         'magenta',
    'Myeloid Act.':       'teal',
    'Lymphocyte':         'blue',
    'Endothelial':        'brown',
    'Mural':              'yellow',
    'Unassigned':         'lightgray',
}

category_colors={ct: mcolors.to_hex(c) for ct, c in celltype_color_dict_v1.items()}

imgs_mif_raw_small = pd.read_csv("/projects/varn-lab/USERS/domans/PvR_TME/samplesheet_miq_batch_small.csv", index_col=0)['image']
imgs_mif_raw_large = pd.read_csv("/projects/varn-lab/USERS/domans/PvR_TME/samplesheet_miq_batch_large.csv", index_col=0)['image']
imgs_mif_raw = pd.concat([imgs_mif_raw_small, imgs_mif_raw_large]).sort_index().to_dict()

dpath = '/projects/varn-lab/USERS/domans/PvR_TME/results-miq-all-v6'
slides = ['CD25016', 'CD25017', 'CD25018', 'CD25019', 'CD25020', 'CD25036', 'CD25043', 'CD25044',
        'CD25045', 'CD25046', 'CD25047', 'CD25084', 'CD25085', 'CD25086', 'CD25087', 'CD25088',
        'CD26001', 'CD26002', 'CD26003', 'CD26004', 'CD26005', 'CD26006', 'CD26007', 'CD26008',
        'CD26010', 'CD26012', 'CD26016', 'CD26017', 'CD26018', 'CD26019', 'CD26020', 'CD26021']
for sample in tqdm(slides, desc='Preparing cell indices'):
    sname = f'{dpath}/{sample}/index.pklz'
    if not os.path.isfile(sname):
        index = pd.Index(pd.read_parquet(f'{dpath}/{sample}/cell_boundaries.parquet')['cell_id']) + '.' + sample
        index = index[~index.duplicated(keep='first')]
        with open(sname, 'wb') as f:
            pickle.dump(index, f)

sys.path.append('/projects/varn-lab/USERS/domans/PvR_TME/corrector/lib')
from meta import samples as samples_mapping

In [ ]:
classifierPaths = 'classifiers/'
dianne.setupClassifierPaths(classifierPaths)

samples = ['CD25016', 'CD25018', 'CD25020']
mpath = '/projects/varn-lab/USERS/domans/PvR_TME/histology/SenReg-registration-results-32'
for sample in samples:
    createH2(sample, mpath)

dataPathSOT = '/projects/varn-lab/USERS/domans/PvR_TME/results-miq-all-v6'
xenium_bundle_paths = {sample: f'{dataPathSOT}/{sample}/' for sample in samples}
xenium_to_he_matrices = {sample: idm for sample in samples}

reload = False
cache_name = f'{classifierPaths}/annotation-v3.pklz'
if not os.path.isfile(cache_name) or reload:
    print("Reloading annotations")
    annotations = {}
    for sample in tqdm(samples, desc='Loading celltype annotations'):
        with open(f'{dataPathSOT}/{sample}/index.pklz', 'rb') as f:
            index = pickle.load(f)
        se_celltype = pd.read_csv(f'{dataPathSOT}/{sample}/celltype-annotation.csv', index_col=0).iloc[:, 0]
        annotations[sample] = pd.Categorical(se_celltype[index].values)
    with open(cache_name, 'wb') as tempfile:
        pickle.dump(annotations, tempfile)
else:
    with open(cache_name, 'rb') as tempfile:
        annotations = pickle.load(tempfile)

dataPathSTQ = '/projects/varn-lab/USERS/domans/PvR_TME/histology/STQ-results-32/'
secondary_images = {sample: f'{dataPathSTQ}/{sample}/image.ome.tiff' for sample in samples}
secondary_matrices = {sample: f'{mpath}/{sample}.matrix-H-2.csv' for sample in samples}

enable_histology_annotation = True
if enable_histology_annotation:
    F = 1
    fname = f'img.data.uni2-{F}.h5ad'
    patch_size = 8 # number of tiles, in each dimension, to include in a patch (e.g. 8 means 8x8=64 tiles per patch)
    ts, mpp, tile_size = dianne.loadSTQParams(dataPathSTQ + samples[0], F)
    ads, _, patchCoordinates, patchesCDFs, qs, ts, mpp, L, N = dianne.loadDataAndPreparePatches(samples, dataPathSTQ, fname, L=None, ts=ts, mpp=mpp, N=patch_size)
    runfn = dianne.makeRunFn(patchCoordinates, ads, samples, qs, ts, mpp, tile_size=tile_size, patch_size=patch_size, PCMA_alpha=0.8, alpha_img=0.5, multiplier=2)
    sizes = runfn.sizes
else:
    runfn = None
    sizes = None

drawings = viewer.create_viewer(samples, imgs_mif_raw, draw_on_secondary=True,
                             matrices=xenium_to_he_matrices, xenium_bundle_paths=xenium_bundle_paths, run_inference_fn=runfn,
                             xenium_mpp=0.3250, annotations=annotations, category_colors=category_colors, max_cells=25000, sample_mapping=samples_mapping,
                             secondary_images=secondary_images, secondary_matrices=secondary_matrices, sample_sizes=sizes)[1]